# Reference-guided SN-to-ST annotation

This tutorial uses a labeled macaque single-nucleus RNA sequencing (SN) reference to annotate a macaque T33 spatial transcriptomics (ST) target. BrainBeacon aligns the reference and target gene spaces, keeps pretrained Stage 1 fixed, adapts Stage 2 using reference expression only, and transfers the reference labels to the target in the learned representation space.

| Role | Dataset | Labels used by the workflow |
|---|---|---|
| Reference | Macaque SN | `Class` and `SubClass` |
| Target | Macaque T33 ST | None |

The workflow uses exactly these two expression datasets. Other datasets and target annotations do not define the feature or label space.

## 1. Install and import

Run this notebook from a BrainBeacon checkout with Python 3.10 or later and a CUDA-enabled PyTorch environment. Install the pinned BrainBeacon source, then restart the kernel:

```bash
python -m pip install "git+https://github.com/ChengmingZhang-CAS/BrainBeacon.git@32c719ceb1b0ea774a84e3fd4fa44cd0b24857f3"
```

The full workflow requires a gene dictionary, an ESM gene embedding, and pretrained Stage 1 and Stage 2 checkpoints. See [Pretrained weights](../../README.md#pretrained-weights) for checkpoint setup, and set all local paths in the next section. The code cell below contains the notebook-local utility functions used by the workflow; no separate tutorial Python file is required.

In [ ]:
# Notebook-local utilities. Expand this cell to inspect or modify them.
from datetime import datetime
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse


def start_run(output_dir, configuration):
    """Create a fresh output directory and retain the reader's settings."""
    run = Path(output_dir) / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    run.mkdir(parents=True, exist_ok=False)
    (run / "settings.json").write_text(
        json.dumps(configuration, indent=2, default=str), encoding="utf-8"
    )
    return run


def model_settings(resources, spatial_reference=False):
    """Resolve four model assets; return explicit Stage 1 and Stage 2 settings."""
    for name in ("gene_dict", "esm_embedding", "stage1", "stage2"):
        if not Path(resources[name]).is_file():
            raise FileNotFoundError(f"{name}: {resources[name]}")
    from brainbeacon.configs.config import DEFAULT_PATHS

    DEFAULT_PATHS.update(
        GENE_DICT_PATH=str(resources["gene_dict"]),
        ESM_EMBED_PATH=str(resources["esm_embedding"]),
    )
    stage1 = dict(
        use_gene_id_emb=False,
        use_homo_emb=True,
        use_esm_emb=True,
        use_rna_type_emb=True,
        use_pos_emb=True,
        use_gene_deviation=False,
        use_cell_density=spatial_reference,
        homo_mean_targets=[],
        comp_mean_bias=[],
        comp_center=["esm"],
        comp_scale={"esm": 0.1, "rna": 0.1, "pos": 0.1},
        emb_layer=0,
        use_dev_abs=True,
        use_hvg=False,
        hvg_log1p=False,
        n_hvg=5000,
        strict_load=False,
        force_tokenize=True,
        batch_size=32,
        weight_mode="expression",
        pool_skip_tokens=2,
        gene_dict_path=str(resources["gene_dict"]),
    )
    stage2 = dict(
        max_batch_size=1000,
        max_eval_batch_size=1000,
        use_hidden_pe=spatial_reference,
        pe_weight=0.1 if spatial_reference else 0.0,
        sampling_mode="random",
        use_patch=False,
        eval_use_patch=True,
        stage2_input_mode="raw",
        fit_only=False,
    )
    return (stage1, stage2)


def _map_genes(adata, species, gene_dict):
    """Use the supplied dictionary for deterministic gene mapping; no online lookup."""
    dictionary = gene_dict.var.loc[
        gene_dict.var["species"].astype(str).str.lower() == species.lower()
    ]
    records = []
    for column in ("gene_symbol", "genenames", "ensembl_id", "gene_id"):
        if column in dictionary:
            records.extend(
                zip(
                    dictionary[column].astype(str).str.upper(),
                    dictionary.index.astype(str),
                )
            )
    records.extend(
        zip(dictionary.index.astype(str).str.upper(), dictionary.index.astype(str))
    )
    mapping = pd.DataFrame(records, columns=["alias", "gene"]).drop_duplicates()
    ambiguous = mapping.groupby("alias")["gene"].nunique()
    mapping = mapping.loc[mapping["alias"].isin(ambiguous[ambiguous == 1].index)]
    lookup = mapping.set_index("alias")["gene"].to_dict()
    candidates = [adata.var_names.astype(str)]
    candidates += [
        adata.var[c].astype(str)
        for c in ("ensembl_id", "gene_id", "gene_symbol", "genenames")
        if c in adata.var
    ]
    mapped = pd.Series(index=adata.var_names, dtype=object)
    for values in candidates:
        current = pd.Series(list(values), index=adata.var_names).str.upper().map(lookup)
        mapped = mapped.fillna(current)
    if not mapped.notna().any():
        raise ValueError(
            f"No genes mapped for {species}; check gene IDs and dictionary species."
        )
    out = adata[:, mapped.notna().to_numpy()].copy()
    out.var["original_gene_id"] = out.var_names.astype(str)
    out.var_names = mapped.dropna().astype(str).to_numpy()
    if not out.var_names.is_unique:
        raise ValueError(
            "Multiple input columns map to one gene ID; aggregate duplicate gene columns first."
        )
    if "genenames" not in out.var:
        out.var["genenames"] = out.var["original_gene_id"].astype(str)
    return out


def _read_input(spec, gene_dict, smooth_k):
    import scanpy as sc

    data = ad.read_h5ad(spec["path"])
    if data.X is None or not data.obs_names.is_unique or (not data.var_names.is_unique):
        raise ValueError("Input needs expression .X and unique cell/gene IDs.")
    values = data.X.data if sparse.issparse(data.X) else np.asarray(data.X)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Use finite, non-negative count-like expression in .X.")
    if spec["assay"] != "snrna":
        if "spatial" not in data.obsm:
            for pair in [
                ("spatial1", "spatial2"),
                ("x", "y"),
                ("X", "Y"),
                ("rx", "ry"),
            ]:
                if all((col in data.obs for col in pair)):
                    data.obsm["spatial"] = data.obs[list(pair)].to_numpy(dtype=float)
                    break
        xy = np.asarray(data.obsm.get("spatial", []), dtype=float)
        if xy.shape != (data.n_obs, 2) or not np.isfinite(xy).all():
            raise ValueError('ST input needs finite two-dimensional obsm["spatial"].')
        if smooth_k:
            from brainbeacon.pipeline.cell_label_transfer import manual_spatial_smooth

            manual_spatial_smooth(data, layer_key="smooth", n_neighbors=smooth_k)
            data.X = data.layers["smooth"]
    sc.pp.filter_cells(data, min_genes=max(1, min(200, int(data.n_vars * 0.01))))
    sc.pp.filter_genes(data, min_cells=3)
    return _map_genes(data, spec["species"], gene_dict)


def prepare_pair(
    reference,
    query,
    resources,
    run,
    top_markers=20,
    max_ref_per_label=100,
    smooth_k=0,
    seed=42,
):
    """Prepare exactly two datasets; reference annotations alone define the label space."""
    from brainbeacon.tokenizer import load_gene_dict_and_mean
    from brainbeacon.pipeline.cell_label_transfer import (
        filter_all_by_common_homo,
        select_ref_homo_marker_space,
        sample_cells_by_label,
    )

    (gene_dict, means) = load_gene_dict_and_mean(
        str(resources["gene_dict"]), reference["assay"], use_mean_norm=True
    )
    ref = _read_input(reference, gene_dict, smooth_k)
    qry = _read_input(query, gene_dict, smooth_k)
    label = ref.obs[reference["label"]].astype("string").str.strip()
    class_label = (
        ref.obs[reference.get("class_label", reference["label"])]
        .astype("string")
        .str.strip()
    )
    invalid = {"", "nan", "none", "unknown", "unassigned", "na", "n/a", "null"}
    valid = (
        label.notna()
        & class_label.notna()
        & ~label.str.lower().isin(invalid)
        & ~class_label.str.lower().isin(invalid)
    )
    ref = ref[valid.to_numpy()].copy()
    ref.obs["label"] = label[valid].astype(str).to_numpy()
    ref.obs["class_label"] = class_label[valid].astype(str).to_numpy()
    if ref.obs["label"].nunique() < 2:
        raise ValueError("Reference must contain at least two labeled cell types.")
    (pair, common, _) = filter_all_by_common_homo(
        {"reference": ref, "query": qry},
        {"reference": reference["species"], "query": query["species"]},
        gene_dict,
    )
    available = set(pair["reference"].var["homo_connect_id"].astype(str))
    mode = (
        "hier"
        if reference.get("class_label") and ref.obs["class_label"].nunique() > 1
        else "subclass"
    )
    (selected, per_label) = select_ref_homo_marker_space(
        pair["reference"],
        available,
        gene_dict,
        means,
        top_markers,
        selection_mode=mode,
        marker_score_mode="second",
        min_score=0.0,
        class_label_col="class_label",
        subclass_label_col="label",
    )
    folder = Path(run) / "prepared"
    folder.mkdir(exist_ok=False)
    selected.to_csv(folder / "selected_homolog_groups.csv", index=False)
    per_label.to_csv(folder / "markers_by_label.csv", index=False)
    common.to_csv(folder / "common_homolog_summary.csv", index=False)
    (paths, rows) = ({}, [])
    expected = set(ref.obs["label"])
    for name, data in pair.items():
        data = data[
            :,
            data.var["homo_connect_id"]
            .astype(str)
            .isin(selected["homo_connect_id"].astype(str)),
        ].copy()
        expressed = np.asarray((data.X > 0).sum(axis=1)).ravel()
        data = data[expressed >= 3].copy()
        if name == "reference":
            (data, _) = sample_cells_by_label(
                data, "label", max_ref_per_label, seed=seed
            )
            if set(data.obs["label"]) != expected:
                raise ValueError(
                    "Gene/cell filtering removed a reference type; inspect selected markers."
                )
        if data.n_obs == 0 or data.n_vars < 3:
            raise ValueError(f"{name}: too few cells or genes remain after filtering.")
        paths[name] = folder / f"{name}.h5ad"
        data.write_h5ad(paths[name], compression="gzip")
        rows.append({"dataset": name, "cells": data.n_obs, "genes": data.n_vars})
    summary = pd.DataFrame(rows).set_index("dataset")
    summary.to_csv(folder / "summary.csv")
    return (paths, summary)


def fit_reference(
    prepared,
    reference,
    resources,
    run,
    stage1,
    stage2,
    global_rounds=10,
    epochs_per_round=50,
    seed=42,
):
    """Freeze Stage 1 via the upstream inference pipeline; fit Stage 2 on reference only."""
    import torch
    from brainbeacon.pipeline.cell_embedding import train_stage2_on_multi_adata

    if not torch.cuda.is_available():
        raise RuntimeError("Full model training requires a CUDA GPU environment.")
    path = Path(prepared["reference"])
    dataset = dict(
        data_name="reference",
        data_dir=str(path.parent),
        adata_name=path.name,
        species=reference["species"],
        assay=reference["assay"],
    )
    checkpoint = train_stage2_on_multi_adata(
        dataset_info_list=[dataset],
        gene_dict_path=str(resources["gene_dict"]),
        stage1_ckpt_path=str(resources["stage1"]),
        stage2_ckpt_path=str(resources["stage2"]),
        output_dir=str(Path(run) / "model"),
        output_prefix="reference_stage2",
        stage1_config=stage1,
        stage2_config=stage2,
        num_global_epochs=global_rounds,
        per_dataset_epochs=epochs_per_round,
        shuffle_each_epoch=False,
        use_single_slice=True,
        stage1_mode="memory",
        fit_only=True,
        device="cuda",
        seed=seed,
        deterministic=True,
        save_model=True,
        save_all_epochs=False,
    )
    checkpoint = Path(checkpoint).resolve()
    if not checkpoint.is_file() or not checkpoint.is_relative_to(Path(run).resolve()):
        raise RuntimeError(
            "Training did not produce a checkpoint in this run directory."
        )
    return checkpoint


def embed_pair(
    prepared, reference, query, resources, checkpoint, run, stage1, stage2, seed=42
):
    """Use the same newly fitted checkpoint for both datasets; do not train on query."""
    from brainbeacon.pipeline.cell_embedding import run_brainbeacon_pipeline

    result = {}
    for name, spec in [("reference", reference), ("query", query)]:
        data = run_brainbeacon_pipeline(
            adata_path=str(prepared[name]),
            species=spec["species"],
            assay=spec["assay"],
            gene_dict_path=str(resources["gene_dict"]),
            stage1_ckpt_path=str(resources["stage1"]),
            stage2_ckpt_path=str(checkpoint),
            output_dir=str(Path(run) / "embeddings" / name),
            output_prefix=name,
            stage1_config=stage1,
            stage2_config={**stage2, "fit_only": False},
            do_fit=False,
            fit_epochs=0,
            use_single_slice=True,
            stage1_mode="memory",
            save_model=False,
            device="cuda",
            seed=seed,
            deterministic=True,
        )
        original = ad.read_h5ad(prepared[name], backed="r")
        try:
            if not data.obs_names.equals(original.obs_names):
                raise ValueError(
                    f"{name}: embedding output changed cell IDs or their order."
                )
        finally:
            original.file.close()
        values = np.asarray(data.obsm["X_emb"])
        if (
            values.ndim != 2
            or values.shape[0] != data.n_obs
            or (not np.isfinite(values).all())
        ):
            raise ValueError(f"{name}: invalid X_emb.")
        result[name] = data
    return (result["reference"], result["query"])


def transfer_labels(reference, query, k=30):
    """Within-dataset centering followed by cosine distance-weighted KNN."""
    from sklearn.neighbors import KNeighborsClassifier

    x_ref = np.asarray(reference.obsm["X_emb"], dtype=np.float32)
    x_query = np.asarray(query.obsm["X_emb"], dtype=np.float32)
    if (
        not 1 <= k <= reference.n_obs
        or not np.isfinite(x_ref).all()
        or (not np.isfinite(x_query).all())
    ):
        raise ValueError("Check k and finite embeddings.")
    if reference.obs["label"].isna().any():
        raise ValueError("Reference labels contain missing values.")
    x_ref = x_ref - x_ref.mean(axis=0, keepdims=True)
    x_query = x_query - x_query.mean(axis=0, keepdims=True)
    model = KNeighborsClassifier(
        n_neighbors=k, metric="cosine", weights="distance", n_jobs=-1
    )
    return model.fit(x_ref, reference.obs["label"].astype(str)).predict(x_query)


def save_predictions(query, labels, run):
    """Export IDs, coordinates, predictions and a compact H5AD without expression .X."""
    xy = np.asarray(query.obsm["spatial"])
    table = pd.DataFrame(
        {"cell_id": query.obs_names, "x": xy[:, 0], "y": xy[:, 1], "prediction": labels}
    )
    table.to_csv(Path(run) / "predictions.tsv.gz", sep="\t", index=False)
    compact = ad.AnnData(
        obs=pd.DataFrame(
            {"prediction": np.asarray(labels).astype(str)}, index=query.obs_names.copy()
        )
    )
    compact.obsm["spatial"] = xy.copy()
    compact.obsm["X_emb"] = np.asarray(query.obsm["X_emb"]).copy()
    compact.write_h5ad(Path(run) / "query_predictions.h5ad", compression="gzip")
    return table


def plot_predictions(
    table, run, title="Predicted cell types", palette=None, invert_y=False
):
    """Draw spatial labels and composition with the same colors; save PNG/PDF/SVG."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    counts = table["prediction"].value_counts()
    labels = sorted(counts.index.astype(str))
    palette = dict(palette or {})
    colors = list(plt.get_cmap("tab20").colors[::2]) + list(
        plt.get_cmap("tab20b").colors
    )
    for i, label in enumerate(labels):
        palette.setdefault(
            label,
            colors[i] if i < len(colors) else plt.get_cmap("hsv")(i / len(labels)),
        )
    plt.rcParams.update({"pdf.fonttype": 42, "svg.fonttype": "none"})
    figures = []
    (fig, ax) = plt.subplots(figsize=(7, 8))
    ax.scatter(
        table.x,
        table.y,
        c=table.prediction.map(palette).tolist(),
        s=0.08 if len(table) > 100000 else 1.2,
        linewidths=0,
        rasterized=True,
    )
    ax.set(aspect="equal", title=title)
    if invert_y:
        ax.invert_yaxis()
    ax.axis("off")
    ax.legend(
        handles=[
            Line2D(
                [],
                [],
                marker="o",
                linestyle="",
                markersize=4,
                color=palette[label],
                label=label,
            )
            for label in labels
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=4,
        frameon=False,
        fontsize=8,
    )
    figures.append(("spatial_predictions", fig))
    (fig, ax) = plt.subplots(figsize=(9, 4))
    percent = counts.reindex(labels) / counts.sum() * 100
    ax.bar(labels, percent, color=[palette[x] for x in labels])
    ax.set(ylabel="Predicted observations (%)", title=title)
    ax.tick_params(axis="x", labelrotation=60)
    figures.append(("predicted_composition", fig))
    for name, fig in figures:
        for suffix in ("png", "pdf", "svg"):
            fig.savefig(Path(run) / f"{name}.{suffix}", dpi=200, bbox_inches="tight")
    counts.rename_axis("prediction").rename("observations").to_csv(
        Path(run) / "composition.csv"
    )
    return [fig for (_, fig) in figures]

## 2. Paths and input configuration

Set `PROJECT_ROOT` to the BrainBeacon repository root and place or link the two example AnnData files under `INPUT_DIR`. Each input must contain non-negative, count-like expression in `.X` and unique cell and gene identifiers. Spatial transcriptomic inputs must provide finite two-dimensional coordinates in `.obsm['spatial']` or in a supported pair of `.obs` columns.

Gene identifiers or symbols are mapped with the supplied species-specific gene dictionary; no online mapping is performed. The reference must contain the configured label column. Target labels are neither required nor read. With the default `SMOOTH_K=10`, target ST expression is replaced by a 10-nearest-neighbor Gaussian-smoothed layer before filtering and gene mapping; set `SMOOTH_K=0` to disable this step.

In [ ]:
from pathlib import Path

# Edit this path for your local BrainBeacon checkout.
PROJECT_ROOT = Path("/path/to/BrainBeacon").expanduser().resolve()
INPUT_DIR = PROJECT_ROOT / "data" / "cell_label_transfer"
OUTPUT_DIR = (
    PROJECT_ROOT / "downstream_tasks" / "cell_label_transfer" / "outputs" / "sn_to_st"
)

REFERENCE = dict(
    path=INPUT_DIR / "macaque1_snRNA_filtered1000.h5ad",
    species="macaque",
    assay="snrna",
    label="SubClass",
    class_label="Class",
)
QUERY = dict(
    path=INPUT_DIR / "Macaque_T33_rowcounts.filtered.consensus_labels.h5ad",
    species="macaque",
    assay="stereo",
)
RESOURCES = dict(
    gene_dict=PROJECT_ROOT / "prior_knowledge" / "gene_dict.h5ad",
    esm_embedding=PROJECT_ROOT / "prior_knowledge" / "esm2_embeddings_d5120.pt",
    stage1=PROJECT_ROOT / "pretrained" / "stage1_fix_step_800000.pt",
    stage2=PROJECT_ROOT / "pretrained" / "stage2_ep280_200.pt",
)

REFERENCE_NAME = "Macaque SN"
QUERY_NAME = "Macaque T33 ST"
INVERT_Y = True
TOP_MARKERS = 20
MAX_REF_PER_LABEL = 100
SMOOTH_K = 10

# Full tutorial settings; reduce these values only for an initial smoke test.
GLOBAL_ROUNDS, EPOCHS_PER_ROUND = 10, 50
KNN_K, SEED = 30, 42

## 3. Create a run and load model settings

Each run uses a new output directory. Stage 1 uses the pretrained homolog and ESM representations. For this SN reference, cell-density features and Stage 2 hidden positional encoding are disabled. You can inspect or edit the returned dictionaries before training.

In [ ]:
stage1, stage2 = model_settings(
    RESOURCES, spatial_reference=REFERENCE["assay"] != "snrna"
)
run = start_run(
    OUTPUT_DIR,
    dict(
        reference=REFERENCE,
        query=QUERY,
        resources=RESOURCES,
        stage1=stage1,
        stage2=stage2,
        top_markers=TOP_MARKERS,
        max_ref_per_label=MAX_REF_PER_LABEL,
        smooth_k=SMOOTH_K,
        global_rounds=GLOBAL_ROUNDS,
        epochs_per_round=EPOCHS_PER_ROUND,
        k=KNN_K,
        seed=SEED,
    ),
)
print(f"Run directory: {run}")

## 4. Prepare the two datasets
The preparation step checks expression and coordinates, maps genes with the fixed dictionary, keeps shared homolog groups, and selects reference markers (up to `TOP_MARKERS` groups per label/level). All genes belonging to a selected group are retained. The two species need not have the same number of genes.

Cells must express at least three selected genes. When `MAX_REF_PER_LABEL > 0`, reference cells are capped per label; a value of 0 keeps all surviving reference cells. All surviving target cells are retained, and target labels are never read. Inspect the returned cell/gene counts and `prepared/markers_by_label.csv` before continuing.

In [ ]:
prepared, summary = prepare_pair(
    REFERENCE,
    QUERY,
    RESOURCES,
    run,
    top_markers=TOP_MARKERS,
    max_ref_per_label=MAX_REF_PER_LABEL,
    smooth_k=SMOOTH_K,
    seed=SEED,
)
summary

## 5. Keep Stage 1 fixed; adapt Stage 2 on the reference
Only the prepared reference enters training. Stage 2 starts from its pretrained checkpoint. Each global round continues the previous weights and creates a new optimizer; the default is 10 × 50 epochs. The upstream training function may cap reference cells internally. Internal validation loss is a training diagnostic, not independent label-transfer accuracy.

In [ ]:
checkpoint = fit_reference(
    prepared,
    REFERENCE,
    RESOURCES,
    run,
    stage1,
    stage2,
    global_rounds=GLOBAL_ROUNDS,
    epochs_per_round=EPOCHS_PER_ROUND,
    seed=SEED,
)
print("Stage 2 checkpoint:", checkpoint)

## 6. Generate reference and target embeddings
Both datasets use the fixed Stage 1 and the same newly fitted Stage 2 checkpoint. Inference sets `do_fit=False`, so the target does not update model weights. `X_emb` is the Stage 2 representation; it is different from tissue spatial coordinates.

In [ ]:
ref_emb, query_emb = embed_pair(
    prepared,
    REFERENCE,
    QUERY,
    RESOURCES,
    checkpoint,
    run,
    stage1,
    stage2,
    seed=SEED,
)
print(
    "Reference/query embedding shapes:",
    ref_emb.obsm["X_emb"].shape,
    query_emb.obsm["X_emb"].shape,
)

## 7. Transfer labels and save results
Center the reference and target embeddings separately, then use cosine distance-weighted kNN. Predictions belong to the reference vocabulary. Save observation IDs, coordinates and labels to a compressed TSV and a compact H5AD. The H5AD includes `X_emb` and spatial coordinates, but no expression `.X`.

In [ ]:
predictions = transfer_labels(ref_emb, query_emb, k=KNN_K)
table = save_predictions(query_emb, predictions, run)
table.head()

## 8. Visualize the predictions
Use the same colors for the spatial map and composition chart. PNG, PDF and SVG files plus a composition CSV are saved automatically. Dense scatter points are rasterized inside PDF/SVG; labels remain text. These plots show label distribution, not prediction accuracy.

In [ ]:
figures = plot_predictions(
    table, run, title=f"{REFERENCE_NAME} → {QUERY_NAME}", invert_y=INVERT_Y
)
plt.show()
print(
    "Prediction files:",
    run / "query_predictions.h5ad",
    run / "predictions.tsv.gz",
)

## Notes

- Stage 1 remains fixed throughout the workflow.
- Stage 2 is adapted with reference expression only; the target is used for inference only.
- Before homolog-group alignment, cells with too few detected genes and genes detected in fewer than three cells are removed.
- `X_emb` is the learned Stage 2 representation and is distinct from the tissue coordinates in `.obsm['spatial']`.
- Cosine distance-weighted k-nearest-neighbor matching uses only reference labels. The resulting plots show predicted label distributions, not independent accuracy estimates.
- The default training settings require a CUDA GPU. Runtime and memory use depend on reference size; `MAX_REF_PER_LABEL` can cap reference observations per label during preparation.